# Histograma de candidatos a decaimiento de muones

Este notebook toma `salida_decaimientos/resumen.csv`, evita contar varias veces el mismo evento cuando aparece en más de un canal, calcula un único $\Delta t$ por evento y grafica $\Delta t$ vs número de cuentas.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Configuración

Este notebook está pensado para correrse desde la carpeta raíz `muones/`.


In [ ]:
RUTA_RESUMEN = Path('salida_decaimientos/resumen.csv')
RUTA_FIGURA = Path('salida_decaimientos/histograma_decaimientos.pdf')

# ancho de bin en microsegundos
BIN_WIDTH_US = 0.25

# rango temporal a mostrar
DT_MIN_US = 0.0
DT_MAX_US = None


## Cargar el resumen de candidatos


In [ ]:
df = pd.read_csv(RUTA_RESUMEN)

print('Columnas encontradas:')
print(df.columns.tolist())

print('\nNúmero de filas del resumen:')
print(len(df))

df.head()


## Evitar contar dos veces el mismo evento

Un evento físico se identifica por el par `(archivo, evento)`. Si el mismo candidato aparece en dos o tres canales, se promedian los valores de `dt_ns` y se cuenta una sola vez.


In [ ]:
eventos = (
    df.groupby(['archivo', 'evento'], as_index=False)
      .agg(
          dt_ns=('dt_ns', 'mean'),
          n_canales=('dt_ns', 'count')
      )
)

print('Número de eventos físicos únicos:')
print(len(eventos))

print('\nCantidad de eventos según número de canales que los detectaron:')
print(eventos['n_canales'].value_counts().sort_index())

eventos.head()


## Convertir $\Delta t$ a microsegundos


In [ ]:
eventos['dt_us'] = eventos['dt_ns'] / 1000.0

mask = np.isfinite(eventos['dt_us'])

if DT_MIN_US is not None:
    mask &= eventos['dt_us'] >= DT_MIN_US

if DT_MAX_US is not None:
    mask &= eventos['dt_us'] <= DT_MAX_US

eventos_filtrados = eventos[mask].copy()
dt_us = eventos_filtrados['dt_us'].to_numpy()

print('Eventos usados en el histograma:', len(dt_us))

if len(dt_us) == 0:
    raise RuntimeError('No quedaron eventos válidos para construir el histograma.')


## Histograma de tiempos de decaimiento


In [ ]:
dt_min = DT_MIN_US if DT_MIN_US is not None else dt_us.min()
dt_max = DT_MAX_US if DT_MAX_US is not None else dt_us.max()

bins = np.arange(
    dt_min,
    dt_max + BIN_WIDTH_US,
    BIN_WIDTH_US
)

fig, ax = plt.subplots(figsize=(8, 5))

cuentas, bordes, _ = ax.hist(
    dt_us,
    bins=bins,
    edgecolor='black'
)

ax.set_xlabel(r'$\Delta t$ [$\mu$s]')
ax.set_ylabel('Número de cuentas')

plt.tight_layout()
plt.savefig(RUTA_FIGURA, bbox_inches='tight')
plt.show()


## Resumen numérico


In [ ]:
print(f'Delta t mínimo = {dt_us.min():.3f} us')
print(f'Delta t máximo = {dt_us.max():.3f} us')
print(f'Delta t medio  = {dt_us.mean():.3f} us')
print(f'Figura guardada en: {RUTA_FIGURA}')
